# V3-4A — A2-MP-HN1: frozen ResNet18 + mean-max pooling

این notebook اولین baseline مرحلهٔ V3-4 است. از hard negativeهای V3-3 با وزن 1.5 استفاده می‌کند، اما encoder را فریز نگه می‌دارد. هدف، مقایسهٔ تمیز با A2-MP است؛ نه اجرای مدل سنگین.

**پیش از اجرای طولانی:** فقط یک kernel باید این notebook را اجرا کند. feature extraction قابل ادامه است.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys
from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SCRIPTS_DIR = PROJECT_ROOT / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from v3_a2mp_hn1 import Config, build_context, cache_preflight, context_report, ensure_feature_cache, train_head, evaluate_best_model

config = Config()
# ابتدا preflight را اجرا کن. فقط وقتی هیچ اجرای تکراری فعال نیست، این مقدار را True کن.
RUN_FULL_FEATURE_CACHE = False
# برای smoke test می‌توان مثلاً 10 گذاشت؛ برای اجرای کامل None بماند.
MAX_NEW_SEQUENCES = None
RUN_TRAINING_AFTER_COMPLETE_CACHE = True

print({'data_root': str(config.data_root), 'device_policy': 'cuda if available else cpu', 'run_full_feature_cache': RUN_FULL_FEATURE_CACHE, 'epochs': config.epochs, 'batch_size': config.batch_size})

{'data_root': 'P:\\NexarCollisionData', 'device_policy': 'cuda if available else cpu', 'run_full_feature_cache': False, 'epochs': 30, 'batch_size': 64}


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1) Freeze V3-4A input selection: positive cores + balanced normal negatives + V3-3 hard negatives.
context = build_context(config)
report = context_report(context)
print(report)
display(context['train_rows'].groupby(['training_role', 'video_label']).agg(windows=('sequence_id', 'size'), videos=('video_id', 'nunique'), loss_weight=('loss_weight', 'first')))
assert context['train_rows'].split.eq('train').all()
assert context['validation_rows'].split.eq('validation').all()
assert not set(context['train_rows'].video_id) & set(context['validation_rows'].video_id)


{'device': 'cpu', 'run_signature': '9651fabfefd53d08b8a6af3164423d94a194407e7c3e2cce6aa14a31c48c301a', 'train_rows': 1446, 'train_role_counts': {'normal_negative': 603, 'positive_core': 603, 'hard_negative_r1': 240}, 'train_video_label_counts': {0: 843, 1: 603}, 'validation_windows_all': 1768, 'validation_videos': 120, 'positive_loss_weight': 1.5970149253731343, 'train_manifest_path': 'P:\\NexarCollisionData\\manifests_v3\\a2mp_hn1_train_windows.csv'}


,,windows,videos,loss_weight
training_role,video_label,,,
hard_negative_r1,0,240,99,1.500000
normal_negative,0,603,240,1.000000
positive_core,1,603,240,1.597015


In [3]:
# 2) Two-window preflight. This confirms RGB, letterbox and the frozen ResNet18 feature shape.
preflight = cache_preflight(context)
print(preflight)


{'preflight_rows': 2, 'preflight_seconds': 25.178765399999975, 'estimated_minutes_missing_only': 497.4904396949995, 'feature_shape': (32, 512), 'reused_hard_negative_features': 843, 'decode_statuses': ['exact', 'exact']}


In [4]:
# 3) Resumable cache. With False it only reuses the 240 hard-negative features and reports the remaining work.
# With True it decodes only missing sequences, writes a partial cache every 25 batches, then produces the final cache.
cache_result = ensure_feature_cache(context, run_full_cache=RUN_FULL_FEATURE_CACHE, max_sequences=MAX_NEW_SEQUENCES)
print({key: value for key, value in cache_result.items() if key not in {'features_by_sequence', 'feature_source_by_sequence'}})
display(pd.Series(cache_result['feature_source_by_sequence']).value_counts().rename_axis('feature_source').to_frame('sequences'))
if not cache_result['complete']:
    print('Cache is intentionally incomplete. Set RUN_FULL_FEATURE_CACHE=True and Run All only when you are ready for the long resumable extraction.')


{'features_available': 843, 'features_expected': 3214, 'missing_after_run': 2371, 'failures_this_run': 0, 'complete': False, 'new_decode_requested': False, 'new_decode_rows_this_run': 2371}


,sequences
feature_source,
reused_v3_3_hard_negative_cache,843


Cache is intentionally incomplete. Set RUN_FULL_FEATURE_CACHE=True and Run All only when you are ready for the long resumable extraction.


In [5]:
# 4) Train only the A2-MP mean-max head after the cache is complete.
training_result = None
if cache_result['complete'] and RUN_TRAINING_AFTER_COMPLETE_CACHE:
    training_result = train_head(context, cache_result)
    print(training_result)
else:
    print('Training has not started: the complete feature cache is required first.')


Training has not started: the complete feature cache is required first.


In [6]:
# 5) Full-MP4-aligned validation: score every V3 validation window and compare fixed aggregation rules.
evaluation_result = None
if cache_result['complete'] and config.best_model_path.is_file():
    evaluation_result = evaluate_best_model(context, cache_result)
    print(evaluation_result['summary']['primary_metrics_validation_selected_threshold'])
    display(evaluation_result['aggregation_ablation'])
else:
    print('Evaluation waits for the complete cache and trained A2-MP-HN1 checkpoint.')


Evaluation waits for the complete cache and trained A2-MP-HN1 checkpoint.


## اجرای کامل

پس از بررسی preflight، فقط `RUN_FULL_FEATURE_CACHE = True` را تنظیم کن و Run All بزن. در اجرای بعدی، cache از همان نقطه ادامه می‌یابد. پس از تولید نتیجه، معیار اصلی این notebook F1 و Recall در سطح MP4 با aggregation ثابت `top3_mean` است؛ سایر aggregationها فقط ablation ثبت‌شده‌اند.